<a href="https://colab.research.google.com/github/idoschw3/Corpus2GeoRAG/blob/main/ApplyNER/NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [80]:
from transformers import pipeline
from google.colab import userdata
import pandas as pd
import numpy as np
import subprocess
import json
import os

### Clone Corpus2GeoRAG Repository

In [81]:
%%bash
rm -rf Corpus2GeoRAG
git clone https://github.com/EtzionR/Corpus2GeoRAG.git

Cloning into 'Corpus2GeoRAG'...


### Data Loading and Initial Inspection

**Purpose:** To load the raw text data, which is crucial for subsequent NER tasks, from a JSON file. This section also includes an example to visually inspect the structure and content of a single entry, ensuring the data is correctly loaded and accessible.


**Input/Output:** Input is `/content/Corpus2GeoRAG/examples/DATA.json`.

**Date:** 2026-09-23

In [82]:
PATH = r'/content/Corpus2GeoRAG/examples/DATA.json'
PATH

'/content/Corpus2GeoRAG/examples/DATA.json'

In [83]:
with open(PATH, 'r') as file:
    data = json.load(file)

len(data)

897

In [84]:
[*data.keys()][:10]

['12 (song)',
 '14 May 2026 Russian strikes on Ukraine',
 '15th BRICS summit',
 '16 March 2022 Chernihiv breadline attack',
 '17 November 2024 Russian strikes on Ukraine',
 '18 March 2022 Mykolaiv military quarters attack',
 '1st European Political Community Summit',
 '2000 Meters to Andriivka',
 '2014 pro-Russian unrest in Ukraine',
 '2020s European re-armament']

In [85]:
example = np.random.choice([*data],1)[0]


print(f'PAGE: {example}:\n\n{data[example]}')


PAGE: Boycott of Russia and Belarus:

Since early 2022, Russia and Belarus have been boycotted by many companies and organizations in Europe, North America, Australasia, and elsewhere, in response to the Russian invasion of Ukraine, which is supported by Belarus. As of 2 July 2022, the Yale School of Management recorded more than 1,000 companies withdrawing or divesting themselves from Russia, either as a result of sanctions or in protest of Russian actions. Ukrainian National Agency on Corruption Prevention maintains a list called International Sponsors of War that includes companies and individuals still doing business with Russia.


== Overview ==
The majority of countries which sanctioned Russia following its 2014 annexation of Crimea began imposing additional sanctions to punish Russia for invading Ukraine—a move for which Russian President Vladimir Putin had long prepared. Many companies were not impacted by sanctions against Russia but ruled in favour of cutting ties with the co

In [86]:
TITLE = '=='
MIN_LENGTH = 5

paragraphs = data[example].split('\n')
paragraphs_processed = [text for text in paragraphs if text.startswith(TITLE)==False and len(text)>=MIN_LENGTH]

print(f'{len(paragraphs_processed)} Paragraphs Extracted from {len(paragraphs)}\n{round(len(paragraphs_processed)/len(paragraphs)*100,1)}%')

74 Paragraphs Extracted from 168
44.0%


### Text Preprocessing: Paragraph Filtering

**Purpose:** To perform an initial cleaning of the raw text by filtering out structural elements and short, uninformative lines. This step aims to prepare cleaner text segments for downstream NER processing by removing elements such as section titles and empty or minimal lines.

**Date:** 2026-09-24

In [87]:
TITLE = '=='
MIN_LENGTH = 5

data_processed = {}

total_paragraphs = 0
total_processed = 0

for key, text in data.items():
  paragraphs = text.split('\n')
  paragraphs_processed = [text for text in paragraphs if text.startswith(TITLE)==False and len(text)>=MIN_LENGTH]
  data_processed[key] = paragraphs_processed

  total_paragraphs += len(paragraphs)
  total_processed += len(paragraphs_processed)

print(f'{total_processed} paragraphs extracted from {total_paragraphs}\n {round(total_processed/total_paragraphs*100,1)}%')

42822 paragraphs extracted from 88029
 48.6%


In [88]:
example = np.random.choice([*data_processed],1)[0]

print(f'PAGE: {example}:\n\n{data_processed[example]}')

PAGE: Battle of Mala Tokmachka:

['In June 2023, a battle took place around the village of Mala Tokmachka as part of the 2023 Ukrainian counteroffensive and during the Russian invasion of Ukraine.', 'At the beginning of the Russian invasion of Ukraine in 2022, Mala Tokmachka quickly became a frontline settlement. The population would drop to only around 200 people by May 2023 due to the proximity of the fighting to the village and consistent Russian shelling which destroyed much of the local infrastructure and utilities. On 13 June 2023, French researcher Philippe Gros referred to the Ukrainian counteroffensive operations south of Mala Tokmachka as a "major axis", with the intention of advancing in the direction of the occupied city of Melitopol, which had been captured by Russian forces early into the invasion. David Axe of Forbes wrote that "perhaps 10,000 or more" Russian soldiers from the 70th, 291st, 429th, and 503rd Motor Rifle Regiments from the 58th Combined Arms Army\'s 42nd G

In [89]:
SAVE_PATH = '/content/Corpus2GeoRAG/ApplyNER/data_initial_cleaning.json'

with open(SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(data_processed, f, ensure_ascii=False, indent=2)

print(f'Saved to: {SAVE_PATH}')

Saved to: /content/Corpus2GeoRAG/ApplyNER/data_initial_cleaning.json


## Initial NER application

**Purpose:** To perform an initial NER application with base models and analyze the results

**Date:** 2026-09-24

### DistilBERT

In [90]:
MODEL_DistilBERT = "dslim/distilbert-NER"
MODEL_DistilBERT

ner_DistilBERT = pipeline("ner",
               model=MODEL_DistilBERT,
               aggregation_strategy='average')

ner_DistilBERT

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

TokenClassificationPipeline: {'model': 'DistilBertForTokenClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

In [91]:
output_DistilBERT = ner_DistilBERT(data_processed[example])
output_DistilBERT

[[{'entity_group': 'LOC',
   'score': np.float32(0.9759275),
   'word': 'Mala Tokmachka',
   'start': 56,
   'end': 70},
  {'entity_group': 'MISC',
   'score': np.float32(0.9971994),
   'word': 'Ukrainian',
   'start': 91,
   'end': 100},
  {'entity_group': 'MISC',
   'score': np.float32(0.9982047),
   'word': 'Russian',
   'start': 133,
   'end': 140},
  {'entity_group': 'LOC',
   'score': np.float32(0.9971464),
   'word': 'Ukraine',
   'start': 153,
   'end': 160}],
 [{'entity_group': 'MISC',
   'score': np.float32(0.998142),
   'word': 'Russian',
   'start': 24,
   'end': 31},
  {'entity_group': 'LOC',
   'score': np.float32(0.9981012),
   'word': 'Ukraine',
   'start': 44,
   'end': 51},
  {'entity_group': 'LOC',
   'score': np.float32(0.9900874),
   'word': 'Mala Tokmachka',
   'start': 61,
   'end': 75},
  {'entity_group': 'MISC',
   'score': np.float32(0.99874085),
   'word': 'Russian',
   'start': 246,
   'end': 253},
  {'entity_group': 'MISC',
   'score': np.float32(0.9974718)

In [92]:
df_DistilBERT = pd.DataFrame([
    {
        "page": example,
        "paragraph_id": paragraph_id,
        "paragraph": data_processed[example][paragraph_id],
        **entity
    }
    for paragraph_id, entities in enumerate(output_DistilBERT)
    for entity in entities
])

df_DistilBERT

,page,paragraph_id,paragraph,entity_group,score,word,start,end
0,Battle of Mala Tokmachka,0,"In June 2023, a battle took place around the v...",LOC,0.975927,Mala Tokmachka,56,70
1,Battle of Mala Tokmachka,0,"In June 2023, a battle took place around the v...",MISC,0.997199,Ukrainian,91,100
2,Battle of Mala Tokmachka,0,"In June 2023, a battle took place around the v...",MISC,0.998205,Russian,133,140
3,Battle of Mala Tokmachka,0,"In June 2023, a battle took place around the v...",LOC,0.997146,Ukraine,153,160
4,Battle of Mala Tokmachka,1,At the beginning of the Russian invasion of Uk...,MISC,0.998142,Russian,24,31
...,...,...,...,...,...,...,...,...
135,Battle of Mala Tokmachka,12,On 30 June 2023 headmaster Sergeant Major Vale...,ORG,0.663017,Mechanized Brigade,542,560
136,Battle of Mala Tokmachka,12,On 30 June 2023 headmaster Sergeant Major Vale...,PER,0.998674,Mykola Melnyk,602,615
137,Battle of Mala Tokmachka,12,On 30 June 2023 headmaster Sergeant Major Vale...,MISC,0.991855,American,673,681
138,Battle of Mala Tokmachka,12,On 30 June 2023 headmaster Sergeant Major Vale...,MISC,0.754124,M2 Bradleys,690,701


### BERT

In [93]:
MODEL_BERT = "dslim/bert-base-NER"
MODEL_BERT

ner_BERT = pipeline("ner",
               model=MODEL_BERT,
               aggregation_strategy='average')

ner_BERT

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TokenClassificationPipeline: {'model': 'BertForTokenClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

In [94]:
output_BERT = ner_BERT(data_processed[example])
output_BERT

[[{'entity_group': 'LOC',
   'score': np.float32(0.7361562),
   'word': 'Mala Tokmachka',
   'start': 56,
   'end': 70},
  {'entity_group': 'MISC',
   'score': np.float32(0.9995716),
   'word': 'Ukrainian',
   'start': 91,
   'end': 100},
  {'entity_group': 'MISC',
   'score': np.float32(0.9997085),
   'word': 'Russian',
   'start': 133,
   'end': 140},
  {'entity_group': 'LOC',
   'score': np.float32(0.99975055),
   'word': 'Ukraine',
   'start': 153,
   'end': 160}],
 [{'entity_group': 'MISC',
   'score': np.float32(0.9997297),
   'word': 'Russian',
   'start': 24,
   'end': 31},
  {'entity_group': 'LOC',
   'score': np.float32(0.9997954),
   'word': 'Ukraine',
   'start': 44,
   'end': 51},
  {'entity_group': 'LOC',
   'score': np.float32(0.744784),
   'word': 'Mala Tokmachka',
   'start': 61,
   'end': 75},
  {'entity_group': 'MISC',
   'score': np.float32(0.9997429),
   'word': 'Russian',
   'start': 246,
   'end': 253},
  {'entity_group': 'MISC',
   'score': np.float32(0.9997155)

In [95]:
df_BERT = pd.DataFrame([
    {
        "page": example,
        "paragraph_id": paragraph_id,
        "paragraph": data_processed[example][paragraph_id],
        **entity
    }
    for paragraph_id, entities in enumerate(output_BERT)
    for entity in entities
])

df_DistilBERT

,page,paragraph_id,paragraph,entity_group,score,word,start,end
0,Battle of Mala Tokmachka,0,"In June 2023, a battle took place around the v...",LOC,0.975927,Mala Tokmachka,56,70
1,Battle of Mala Tokmachka,0,"In June 2023, a battle took place around the v...",MISC,0.997199,Ukrainian,91,100
2,Battle of Mala Tokmachka,0,"In June 2023, a battle took place around the v...",MISC,0.998205,Russian,133,140
3,Battle of Mala Tokmachka,0,"In June 2023, a battle took place around the v...",LOC,0.997146,Ukraine,153,160
4,Battle of Mala Tokmachka,1,At the beginning of the Russian invasion of Uk...,MISC,0.998142,Russian,24,31
...,...,...,...,...,...,...,...,...
135,Battle of Mala Tokmachka,12,On 30 June 2023 headmaster Sergeant Major Vale...,ORG,0.663017,Mechanized Brigade,542,560
136,Battle of Mala Tokmachka,12,On 30 June 2023 headmaster Sergeant Major Vale...,PER,0.998674,Mykola Melnyk,602,615
137,Battle of Mala Tokmachka,12,On 30 June 2023 headmaster Sergeant Major Vale...,MISC,0.991855,American,673,681
138,Battle of Mala Tokmachka,12,On 30 June 2023 headmaster Sergeant Major Vale...,MISC,0.754124,M2 Bradleys,690,701


## Commit to repo

In [98]:
REPO_PATH = "/content/Corpus2GeoRAG"
GITHUB_USER = "idoschw3"
GITHUB_REPO = "Corpus2GeoRAG"
GITHUB_EMAIL = "193781032+idoschw3@users.noreply.github.com"
COMMIT_MESSAGE = "Update NER workflow"

os.chdir(REPO_PATH)

# Git identity
subprocess.run(["git", "config", "user.name", GITHUB_USER], check=True)
subprocess.run(["git", "config", "user.email", GITHUB_EMAIL], check=True)

# Make sure origin points to my fork
subprocess.run([
    "git", "remote", "set-url", "origin",
    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
], check=True)

# Read GitHub token from Colab Secrets
github_token = userdata.get("GITHUB_TOKEN")

# Temporary authentication helper
askpass_path = "/tmp/github_askpass.sh"

with open(askpass_path, "w") as f:
    f.write(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "$GITHUB_USER" ;;\n'
        '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n'
    )

os.chmod(askpass_path, 0o700)

env = os.environ.copy()
env["GITHUB_USER"] = GITHUB_USER
env["GITHUB_TOKEN"] = github_token
env["GIT_ASKPASS"] = askpass_path
env["GIT_TERMINAL_PROMPT"] = "0"

try:
    # Stage ALL repository changes
    subprocess.run(["git", "add", "-A"], check=True)

    # Commit only if there are changes
    status = subprocess.run(
        ["git", "status", "--porcelain"],
        capture_output=True,
        text=True,
        check=True
    )

    if status.stdout.strip():
        subprocess.run(
            ["git", "commit", "-m", COMMIT_MESSAGE],
            check=True
        )
        print("Changes committed.")
    else:
        print("No new changes to commit.")

    # Get current branch
    branch = subprocess.run(
        ["git", "branch", "--show-current"],
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()

    # Bring in changes already made directly on GitHub
    subprocess.run(
        ["git", "pull", "--rebase", "origin", branch],
        env=env,
        check=True
    )

    print("Local repository synchronized with GitHub.")

    # Push everything back to my fork
    subprocess.run(
        ["git", "push", "origin", branch],
        env=env,
        check=True
    )

    print(f"Successfully pushed to {GITHUB_USER}/{GITHUB_REPO} on branch '{branch}'.")

finally:
    if os.path.exists(askpass_path):
        os.remove(askpass_path)

No new changes to commit.
Local repository synchronized with GitHub.
Successfully pushed to idoschw3/Corpus2GeoRAG on branch 'main'.
